# Episode 13: Training a Network End to End
<a href="https://colab.research.google.com/github/Kinvecta/how-llms-work/blob/main/notebooks/Episode%2013%20-%20Training%20End%20to%20End.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

**Companion notebook** for the series *How Large Language Models Work: From Math to Transformers*.

This notebook reproduces the Math Lab numbers from Episode 13, one step per section. Everything up to the last section uses only `numpy` and the standard library (already installed in Colab), so each cell runs in a second or two. The last section downloads GPT-2 small's weights once (548 MB) to confirm the token IDs and the parameter counts the episode relies on.

Covers: Adam's first and second moves on Episode 12's 2 × 2 network (m, v, bias correction, and what happens without it), a loss scaled by 10, SGD against momentum against Adam, LayerNorm of one vector, the initialization formulas and a seeded ten-layer simulation, one training step's mini-batch arithmetic, nanoGPT's warmup-then-cosine schedule, and the check-question answers.

All numbers are hand-set and illustrative. The toy proves nothing about optimizers at LLM scale. Every value that appears in the episode is checked with an `assert`. If a cell fails, a number in the episode or in the code is wrong.

In [1]:
import math
import numpy as np

NAMES = ["W11", "W12", "W21", "W22", "β1", "β2", "W'11", "W'12", "W'21", "W'22", "β'1", "β'2"]
SUB = {"W21": "W₂₁", "W22": "W₂₂", "β2": "β₂", "W'12": "W′₁₂", "W'22": "W′₂₂", "β'1": "β′₁", "β'2": "β′₂"}
u = np.array([1.0, 2.0])               # the input
y = np.array([9.0, -2.0])              # the target
theta0 = np.array([1, -1, 2, 1, -1, 0, 1, 2, 1, -1, 2, 0], float)   # Episode 12's hand-set parameters

def unpack(t):                         # W, beta, W', beta'
    return t[0:4].reshape(2, 2), t[4:6], t[6:10].reshape(2, 2), t[10:12]

def forward(t):                        # layer 1: u -> h -> psi(h) with psi = ReLU; layer 2: psi(h) -> h'
    W, b, Wp, bp = unpack(t)
    h = W @ u + b
    a = np.maximum(h, 0.0)
    hp = Wp @ a + bp
    return h, a, hp, float(np.sum((hp - y) ** 2))

def blame(t):                          # backward pass, output first (the calculus chain rule at every operation)
    W, b, Wp, bp = unpack(t)
    h, a, hp, _ = forward(t)
    dhp = 2 * (hp - y)                 # blame on h'
    dWp = np.outer(dhp, a)             # blame on W'
    da = Wp.T @ dhp                    # blame on psi(h)
    dh = da * (h > 0)                  # ReLU's slope: 1 where h > 0, else 0
    dW = np.outer(dh, u)
    return np.concatenate([dW.ravel(), dh, dWp.ravel(), dhp])

h0, a0, hp0, loss0 = forward(theta0)
G0 = blame(theta0)
print("h =", h0, " psi(h) =", a0, " h' =", hp0, " loss =", loss0)
print("blame:", {n: float(g) + 0.0 for n, g in zip(NAMES, G0)})        # + 0.0 turns -0.0 into 0.0
assert h0.tolist() == [-2, 4] and a0.tolist() == [0, 4] and hp0.tolist() == [10, -4] and loss0 == 5
assert G0.tolist() == [0, 0, 8, 16, 0, 8, 0, 8, 0, -16, 2, -4]
NONZERO = [i for i in range(12) if G0[i] != 0]
ZERO = [i for i in range(12) if G0[i] == 0]
assert [NAMES[i] for i in NONZERO] == ["W21", "W22", "β2", "W'12", "W'22", "β'1", "β'2"]    # seven parameters have blame
assert [NAMES[i] for i in ZERO] == ["W11", "W12", "β1", "W'11", "W'21"]                      # five have none: neuron 1 is off (h1 = -2)
print("seven parameters with blame, five without")

h = [-2.  4.]  psi(h) = [0. 4.]  h' = [10. -4.]  loss = 5.0
blame: {'W11': 0.0, 'W12': 0.0, 'W21': 8.0, 'W22': 16.0, 'β1': 0.0, 'β2': 8.0, "W'11": 0.0, "W'12": 8.0, "W'21": 0.0, "W'22": -16.0, "β'1": 2.0, "β'2": -4.0}
seven parameters with blame, five without


### Check the blame table by nudging
Central differences on all twelve numbers (the same check as Episode 12). Then one plain SGD step at η = 0.01, the number Episode 13 compares against.

In [2]:
e = 1e-5
for i in range(12):
    tp, tm = theta0.copy(), theta0.copy()
    tp[i] += e; tm[i] -= e
    fd = (forward(tp)[3] - forward(tm)[3]) / (2 * e)
    assert abs(fd - G0[i]) < 1e-6, (NAMES[i], fd, G0[i])
print("all twelve blame numbers match the nudging check")

sgd_1 = forward(theta0 - 0.01 * G0)[3]
print("SGD, eta = 0.01, one step: loss", round(sgd_1, 3))
assert round(sgd_1, 3) == 0.909

all twelve blame numbers match the nudging check
SGD, eta = 0.01, one step: loss 0.909


## Step 1: Adam's first move
η = 0.01, ρ₁ = 0.9, ρ₂ = 0.999, ω = 10⁻⁸, m = v = 0. At k = 1 the corrections divide by 1 − ρ₁ = 0.1 and 1 − ρ₂ = 0.001, so m̂ = blame and v̂ = blame². Each parameter with blame moves by η · blame/(|blame| + ω) = 0.01, whatever its blame; the five without blame do not move.

In [3]:
ETA, RHO1, RHO2, OMEGA = 0.01, 0.9, 0.999, 1e-8

def adam_step(t, m, v, k, g, eta=ETA, rho1=RHO1, rho2=RHO2, omega=OMEGA, correct=True):
    m = rho1 * m + (1 - rho1) * g                      # running average of the blame
    v = rho2 * v + (1 - rho2) * g * g                  # running average of the blame squared
    mh = m / (1 - rho1 ** k) if correct else m         # bias correction
    vh = v / (1 - rho2 ** k) if correct else v
    return t - eta * mh / (np.sqrt(vh) + omega), m, v, mh, vh

def f(x, n):                                           # the episode's number format (true minus sign)
    return f"{x:.{n}f}".replace("-", "−")

t1, m1, v1, mh1, vh1 = adam_step(theta0, np.zeros(12), np.zeros(12), 1, G0)
rows = [f"| {SUB[NAMES[i]]} | {f(G0[i], 0)} | {f(m1[i], 1)} | {f(v1[i], 3)} | {f(theta0[i], 0)} → {f(t1[i], 2)} |" for i in NONZERO]
EXPECTED_STEP1 = ['| W₂₁ | 8 | 0.8 | 0.064 | 2 → 1.99 |', '| W₂₂ | 16 | 1.6 | 0.256 | 1 → 0.99 |', '| β₂ | 8 | 0.8 | 0.064 | 0 → −0.01 |', '| W′₁₂ | 8 | 0.8 | 0.064 | 2 → 1.99 |', '| W′₂₂ | −16 | −1.6 | 0.256 | −1 → −0.99 |', '| β′₁ | 2 | 0.2 | 0.004 | 2 → 1.99 |', '| β′₂ | −4 | −0.4 | 0.016 | 0 → 0.01 |']
for r_ in rows:
    print(r_)
assert rows == EXPECTED_STEP1                          # the table in the episode, row for row
assert abs(1 - RHO1 - 0.1) < 1e-15 and abs(1 - RHO2 - 0.001) < 1e-15
for i in NONZERO:
    assert abs(mh1[i] - G0[i]) < 1e-12 and abs(vh1[i] - G0[i] ** 2) < 1e-9       # m-hat = blame, v-hat = blame squared
    assert abs(abs(t1[i] - theta0[i]) - 0.01) < 1e-10                             # every move is eta
for i in ZERO:
    assert t1[i] == theta0[i] and m1[i] == 0 and v1[i] == 0                       # zero blame: 0/(0 + omega) = 0
loss_a1 = forward(t1)[3]
print("loss after Adam step 1:", round(loss_a1, 6))
assert round(loss_a1, 6) == 4.407224 and round(loss_a1, 3) == 4.407

| W₂₁ | 8 | 0.8 | 0.064 | 2 → 1.99 |
| W₂₂ | 16 | 1.6 | 0.256 | 1 → 0.99 |
| β₂ | 8 | 0.8 | 0.064 | 0 → −0.01 |
| W′₁₂ | 8 | 0.8 | 0.064 | 2 → 1.99 |
| W′₂₂ | −16 | −1.6 | 0.256 | −1 → −0.99 |
| β′₁ | 2 | 0.2 | 0.004 | 2 → 1.99 |
| β′₂ | −4 | −0.4 | 0.016 | 0 → 0.01 |
loss after Adam step 1: 4.407224


## Step 2: bias correction off
Without dividing, m = 0.1 · blame but √v = 0.0316 · |blame|, so every step is 0.1/√0.001 = 3.16 times larger. Here the bigger step helps (a smooth loss, far from its bottom).

In [4]:
tN, mN, vN, _, _ = adam_step(theta0, np.zeros(12), np.zeros(12), 1, G0, correct=False)
for i in NONZERO:
    assert abs(abs(tN[i] - theta0[i]) - 0.031623) < 1e-6          # 3.16 times eta
assert abs(0.1 / math.sqrt(0.001) - 3.1623) < 5e-5 and abs(math.sqrt(0.001) - 0.0316) < 3e-5
loss_nc = forward(tN)[3]
print("loss after one step without bias correction:", round(loss_nc, 4))
assert round(loss_nc, 4) == 3.3079 and round(loss_nc, 3) == 3.308

loss after one step without bias correction: 3.3079


## Step 3: Adam's second move
From the updated network: h = (−2, 3.96), h′ = (9.8704, −3.9104). Now 1 − ρ₁² = 0.19 and 1 − ρ₂² = 0.001999, so m̂ = (0.09 · blame₁ + 0.1 · blame₂)/0.19, weights 0.474 and 0.526.

In [5]:
h1_, a1_, hp1_, _ = forward(t1)
assert [round(x, 4) for x in h1_] == [-2.0, 3.96] and [round(x, 4) for x in hp1_] == [9.8704, -3.9104]
G1 = blame(t1)
assert all(G1[i] == 0 for i in ZERO)                               # still no blame on neuron 1's parameters
t2, m2, v2, mh2, vh2 = adam_step(t1, m1, v1, 2, G1)
rows = []
for i in NONZERO:
    ratio = mh2[i] / (math.sqrt(vh2[i]) + OMEGA)
    rows.append(f"| {SUB[NAMES[i]]} | {f(G1[i], 4)} | {f(m2[i], 5)} | {f(v2[i], 6)} | {f(mh2[i], 5)} | {f(vh2[i], 3)} | {f(ratio, 4)} |")
EXPECTED_STEP3 = ['| W₂₁ | 7.2468 | 1.44468 | 0.116452 | 7.60357 | 58.255 | 0.9962 |', '| W₂₂ | 14.4936 | 2.88936 | 0.465808 | 15.20714 | 233.020 | 0.9962 |', '| β₂ | 7.2468 | 1.44468 | 0.116452 | 7.60357 | 58.255 | 0.9962 |', '| W′₁₂ | 6.8936 | 1.40936 | 0.111457 | 7.41767 | 55.757 | 0.9934 |', '| W′₂₂ | −15.1304 | −2.95304 | 0.484672 | −15.54230 | 242.457 | −0.9982 |', '| β′₁ | 1.7408 | 0.35408 | 0.007026 | 1.86358 | 3.515 | 0.9940 |', '| β′₂ | −3.8208 | −0.74208 | 0.030583 | −3.90568 | 15.299 | −0.9985 |']
for r_ in rows:
    print(r_)
assert rows == EXPECTED_STEP3
assert abs(1 - RHO1 ** 2 - 0.19) < 1e-15 and abs(1 - RHO2 ** 2 - 0.001999) < 1e-12
assert abs(0.9 * 0.1 / 0.19 - 0.47368) < 1e-5 and abs(0.1 / 0.19 - 0.52632) < 1e-5
w21 = NAMES.index("W21")
assert abs(m2[w21] - (0.9 * 0.8 + 0.1 * G1[w21])) < 1e-12 and abs(mh2[w21] - 7.6036) < 5e-5     # the prose's m-hat
assert abs(v2[w21] - (0.999 * 0.064 + 0.001 * G1[w21] ** 2)) < 1e-12 and abs(vh2[w21] - 58.255) < 5e-4
loss_a2 = forward(t2)[3]
print("loss after Adam step 2:", round(loss_a2, 6))
assert round(loss_a2, 6) == 3.869974 and round(loss_a2, 3) == 3.870
for i in ZERO:
    assert t2[i] == theta0[i]

| W₂₁ | 7.2468 | 1.44468 | 0.116452 | 7.60357 | 58.255 | 0.9962 |
| W₂₂ | 14.4936 | 2.88936 | 0.465808 | 15.20714 | 233.020 | 0.9962 |
| β₂ | 7.2468 | 1.44468 | 0.116452 | 7.60357 | 58.255 | 0.9962 |
| W′₁₂ | 6.8936 | 1.40936 | 0.111457 | 7.41767 | 55.757 | 0.9934 |
| W′₂₂ | −15.1304 | −2.95304 | 0.484672 | −15.54230 | 242.457 | −0.9982 |
| β′₁ | 1.7408 | 0.35408 | 0.007026 | 1.86358 | 3.515 | 0.9940 |
| β′₂ | −3.8208 | −0.74208 | 0.030583 | −3.90568 | 15.299 | −0.9985 |
loss after Adam step 2: 3.869974


## Step 4: scale the loss by 10
Every blame is 10 times larger, so SGD's step is too. Adam's m̂ grows 10 times and v̂ 100 times, the ratio is unchanged, and the step with it. This holds while |blame| ≫ ω (here the smallest is 2).

In [6]:
def run(opt, steps, eta, scale=1.0, gamma=0.9, correct=True):
    t = theta0.copy(); m = np.zeros(12); v = np.zeros(12); vel = np.zeros(12)
    hist, moves = [forward(t)[3]], []
    for k in range(1, steps + 1):
        g = scale * blame(t)
        old = t.copy()
        if opt == "sgd":
            t = t - eta * g
        elif opt == "mom":                                   # velocity <- gamma * velocity + eta * blame; theta <- theta - velocity
            vel = gamma * vel + eta * g
            t = t - vel
        else:
            t, m, v, _, _ = adam_step(t, m, v, k, g, eta=eta, correct=correct)
        moves.append(t - old)
        hist.append(forward(t)[3])
    return t, hist, moves

_, hS, mvS = run("sgd", 3, 0.01)
_, hS10, mvS10 = run("sgd", 3, 0.01, scale=10.0)
_, hA, mvA = run("adam", 3, 0.01)
_, hA10, mvA10 = run("adam", 3, 0.01, scale=10.0)
print("after one step: SGD", round(hS[1], 3), "-> x10", round(hS10[1], 1), "| Adam", round(hA[1], 6), "-> x10", round(hA10[1], 6))
assert round(hS[1], 3) == 0.909 and round(hS10[1], 1) == 57.6
assert np.allclose(mvS10[0], 10 * mvS[0], atol=1e-12)                      # SGD's step is 10 times larger
assert max(np.abs(mvA10[k] - mvA[k]).max() for k in range(3)) < 1e-10      # Adam's steps agree to under 1e-10
assert round(hA10[1], 6) == round(hA[1], 6) == 4.407224
assert min(abs(G0[i]) for i in NONZERO) == 2 and 2 / OMEGA > 1e8             # |blame| >> omega for every parameter with blame

after one step: SGD 0.909 -> x10 57.6 | Adam 4.407224 -> x10 4.407224


## Step 5: three optimizers, one network
Loss after k steps from ℓ = 5 (momentum γ = 0.9). One example, 12 parameters, a smooth loss, hand-picked η: the ranking says nothing about optimizers at LLM scale, and η changes it.

In [7]:
_, hSGD, _ = run("sgd", 30, 0.01)
_, hMOM, _ = run("mom", 30, 0.01)
_, hMOM001, _ = run("mom", 30, 0.001)
_, hADAM, _ = run("adam", 30, 0.01)
_, hADAM1, _ = run("adam", 30, 0.1)
COLS = [1, 2, 3, 5, 10, 20]

def srow(name, h):
    return f"| {name} | " + " | ".join("7.7 × 10⁻⁶" if h[k] < 1e-4 else f(h[k], 3) for k in COLS) + " |"

rows = [srow("SGD (0.01)", hSGD), srow("Momentum (0.01)", hMOM), srow("Adam (0.01)", hADAM), srow("Adam (0.1)", hADAM1)]
EXPECTED_STEP5 = ['| SGD (0.01) | 0.909 | 0.482 | 0.262 | 0.077 | 0.004 | 7.7 × 10⁻⁶ |', '| Momentum (0.01) | 0.909 | 1.794 | 2.652 | 1.827 | 1.290 | 0.161 |', '| Adam (0.01) | 4.407 | 3.870 | 3.388 | 2.585 | 1.400 | 0.682 |', '| Adam (0.1) | 1.367 | 1.305 | 1.491 | 0.503 | 0.904 | 0.208 |']
for r_ in rows:
    print(r_)
assert rows == EXPECTED_STEP5
assert abs(hSGD[20] - 7.7e-6) < 5e-8                                       # the 7.7e-6 entry
assert abs(hMOM[1] - hSGD[1]) < 1e-12                                       # momentum's first step is SGD's
assert any(hMOM[k + 1] > hMOM[k] for k in range(1, 4)) and hMOM[1] < hMOM[2] < hMOM[3]    # momentum overshoots: the loss rises from step 1 to 3
assert all(hSGD[k + 1] < hSGD[k] for k in range(20))                       # SGD falls every step
assert hSGD[10] < hMOM[10] < hADAM[10]                                      # in this toy SGD is lowest at step 10
assert hADAM1[20] < hADAM[20]                                               # a larger eta changes the ranking among the Adam runs
assert hMOM001[10] < hMOM[10] and hMOM001[20] > hMOM[20]                    # momentum at 0.001 and 0.01 swap between steps 10 and 20
assert hADAM1[3] > hADAM1[2]                                                # Adam at 0.1 is not monotone: 1.305 then 1.491
# neuron 1 stays off with this one example: its five parameters never move under any optimizer for 30 steps, and h1 stays -2
for opt, eta in (("sgd", 0.01), ("mom", 0.01), ("adam", 0.01), ("adam", 0.1)):
    t_end, _, _ = run(opt, 30, eta)
    assert all(t_end[i] == theta0[i] for i in ZERO) and forward(t_end)[0][0] == -2.0
print("zero-blame parameters never moved (30 steps, four runs)")

| SGD (0.01) | 0.909 | 0.482 | 0.262 | 0.077 | 0.004 | 7.7 × 10⁻⁶ |
| Momentum (0.01) | 0.909 | 1.794 | 2.652 | 1.827 | 1.290 | 0.161 |
| Adam (0.01) | 4.407 | 3.870 | 3.388 | 2.585 | 1.400 | 0.682 |
| Adam (0.1) | 1.367 | 1.305 | 1.491 | 0.503 | 0.904 | 0.208 |
zero-blame parameters never moved (30 steps, four runs)


## Step 6: LayerNorm
h = (0, 2, 2, 8): mean 3, var = ((−3)² + (−1)² + (−1)² + 5²)/4 = 9 (divide by d), √9 = 3, output (−1, −⅓, −⅓, 5/3). Two entries would always give ±(1, −1). ω inside the root, then a gain and an offset.

In [8]:
from fractions import Fraction as Fr

def layernorm(h, gain=None, offset=None, omega=0.0):
    h = np.asarray(h, float)
    d = len(h)
    mean = h.sum() / d
    var = ((h - mean) ** 2).sum() / d                     # divide by d, as Ba et al. do
    z = (h - mean) / np.sqrt(var + omega)                  # omega inside the root
    if gain is not None:
        z = gain * z + offset
    return mean, var, z

mean, var, z = layernorm([0, 2, 2, 8])
print("mean", mean, "var", var, "output", z)
assert mean == 3 and var == 9 and np.allclose(z, [-1, -1 / 3, -1 / 3, 5 / 3])
assert abs(z.mean()) < 1e-15 and abs((z ** 2).mean() - 1) < 1e-15            # mean 0, variance 1
for h2 in ([-2, 4], [0, 7], [5, -9]):                                         # two entries always give +-(1, -1)
    assert np.allclose(np.abs(layernorm(h2)[2]), [1, 1])
_, _, zw = layernorm([0, 2, 2, 8], omega=1e-5)
print("with omega = 1e-5:", round(abs(zw[0]), 7))
assert round(3 / math.sqrt(9 + 1e-5), 7) == 0.9999994 and round(abs(zw[0]), 7) == 0.9999994
_, _, zg = layernorm([0, 2, 2, 8], gain=np.full(4, 2.0), offset=np.full(4, 1.0))
assert np.allclose(zg, [-1, 1 / 3, 1 / 3, 13 / 3])                           # gain 2 and offset 1 on every entry
_, _, z1 = layernorm([0, 2, 2, 8], gain=np.ones(4), offset=np.zeros(4))
assert np.allclose(z1, z)                                                     # nanoGPT's starting gain 1 and offset 0
assert (Fr(36, 3), Fr(36, 4)) == (12, 9)                                      # dividing by d - 1 would give 12, not 9
for s, b in ((10, 0), (1, 100), (10, 10)):                                    # scaling and shifting the input change nothing
    assert np.allclose(layernorm([x * s + b for x in (0, 2, 2, 8)])[2], z)

mean 3.0 var 9.0 output [-1.         -0.33333333 -0.33333333  1.66666667]
with omega = 1e-5: 0.9999994


## Section 2: starting values
Glorot and Bengio's normalized initialization: a uniform draw on ±√6/√(n_in + n_out), variance 2/(n_in + n_out) (the compromise between n_in · Var(W) = 1 and n_out · Var(W) = 1). He et al. for ReLU: Var(W) = 2/n_in. For GPT-2 small's 768 → 3,072 layer.

In [9]:
n_in, n_out = 768, 3072
bound = math.sqrt(6) / math.sqrt(n_in + n_out)
he_std = math.sqrt(2 / n_in)
print("Xavier bound", round(bound, 4), "(std", round(bound / math.sqrt(3), 4), ") | He std", round(he_std, 4))
assert round(bound, 4) == 0.0395 and round(he_std, 4) == 0.0510
assert abs(bound ** 2 / 3 - 2 / (n_in + n_out)) < 1e-15                           # uniform on +-a has variance a^2/3
assert abs(1 / n_in - 0.001302) < 5e-7 and abs(1 / n_out - 0.000326) < 5e-7       # the two one-sided conditions
assert abs(2 / (n_in + n_out) - 0.000521) < 5e-7                                   # their compromise lies between
# a ReLU layer multiplies the variance by n_in * Var(W) / 2; width 256, Xavier: factor 1/2 per layer, 2**-5 after ten
assert abs(256 * (1 / 256) / 2 - 0.5) < 1e-15 and abs((1 / math.sqrt(2)) ** 10 - 2 ** -5) < 1e-15 and 2 ** -5 == 0.03125

Xavier bound 0.0395 (std 0.0228 ) | He std 0.051


### Simulation, not a proof
Width 256, ten ReLU layers, no biases, a unit-scale input and a unit-scale blame seed; one run with NumPy seed 13. The recipe matters: a fresh `default_rng(13)` per row, the ten 256 × 256 matrices drawn first, then the input, then the blame seed, ReLU after every layer. Another seed or draw order moves each figure by tens of percent, so the asserts below allow a factor of about 1.8 from the episode's digits and a factor of 3 from the theory.

In [10]:
def simulate(kind, n=256, depth=10, seed=13):
    rng = np.random.default_rng(seed)                       # a fresh generator for every row
    Ws = []
    for _ in range(depth):                                   # the ten matrices come first
        if kind == "small":
            Ws.append(rng.normal(0, 0.01, (n, n)))
        elif kind == "one":
            Ws.append(rng.normal(0, 1.0, (n, n)))
        elif kind == "xavier":
            a = math.sqrt(6 / (n + n)); Ws.append(rng.uniform(-a, a, (n, n)))
        else:
            Ws.append(rng.normal(0, math.sqrt(2 / n), (n, n)))
    act = rng.normal(0, 1, n)                                # then the input
    masks = []
    for W in Ws:                                             # forward, ReLU after every layer
        hh = W @ act; masks.append(hh > 0); act = np.maximum(hh, 0)
    fwd = math.sqrt(np.mean(act ** 2))
    bl = rng.normal(0, 1, n)                                 # then the blame seed
    for W, mk in zip(Ws[::-1], masks[::-1]):                 # backward through the same masks
        bl = W.T @ (mk * bl)
    return fwd, math.sqrt(np.mean(bl ** 2))

SHOWN = {"small": (3.0e-10, 3.9e-10), "one": (3.0e10, 3.9e10), "xavier": (0.039, 0.033), "he": (0.87, 1.1)}
THEORY = {"small": math.sqrt(256 * 0.01 ** 2 / 2) ** 10, "one": math.sqrt(256 * 1.0 / 2) ** 10,
          "xavier": math.sqrt(256 * (1 / 256) / 2) ** 10, "he": 1.0}
print("theory: small", f"{THEORY['small']:.1e}", "| one", f"{THEORY['one']:.1e}", "| xavier", round(THEORY['xavier'], 5))
assert abs(THEORY["small"] - 3.4e-10) < 0.1e-10 and abs(THEORY["one"] - 3.4e10) < 0.1e10
assert abs(THEORY["xavier"] - 2 ** -5) < 1e-15 and THEORY["he"] == 1.0
sim = {k: simulate(k) for k in SHOWN}
for k, (fw, bw) in sim.items():
    print(f"{k:7s} forward RMS {fw:.2g}   blame RMS {bw:.2g}   (episode: {SHOWN[k][0]:.2g} and {SHOWN[k][1]:.2g})")
    for seed in (13, 14, 15):                                # another seed stays within a factor of 3 of the theory
        f2, b2 = simulate(k, seed=seed)
        assert 0.3 < f2 / THEORY[k] < 3 and 0.3 < b2 / THEORY[k] < 3, (k, seed, f2, b2)
    for got, shown in zip(sim[k], SHOWN[k]):                 # the digits in the episode, within a factor of about 1.8
        assert abs(math.log10(got) - math.log10(shown)) < 0.25, (k, got, shown)
assert sim["he"][0] > 0.3 and sim["he"][1] > 0.3 and sim["xavier"][0] < 0.2 and sim["xavier"][1] < 0.2
# the four rows of the episode's table, as printed
SIMROWS = ['| Normal, std 0.01 | 3.0 × 10⁻¹⁰ | 3.9 × 10⁻¹⁰ |', '| Normal, std 1 | 3.0 × 10¹⁰ | 3.9 × 10¹⁰ |', '| Xavier uniform | 0.039 | 0.033 |', '| He normal | 0.87 | 1.1 |']
assert SIMROWS[0].startswith("| Normal, std 0.01 | 3.0 × 10⁻¹⁰ | 3.9 × 10⁻¹⁰ |") and SIMROWS[1].startswith("| Normal, std 1 | 3.0 × 10¹⁰ | 3.9 × 10¹⁰ |")
assert SIMROWS[2].startswith("| Xavier uniform | 0.039 | 0.033 |") and SIMROWS[3].startswith("| He normal | 0.87 | 1.1 |")

theory: small 3.4e-10 | one 3.4e+10 | xavier 0.03125
small   forward RMS 3e-10   blame RMS 3.9e-10   (episode: 3e-10 and 3.9e-10)
one     forward RMS 3e+10   blame RMS 3.9e+10   (episode: 3e+10 and 3.9e+10)


xavier  forward RMS 0.039   blame RMS 0.033   (episode: 0.039 and 0.033)


he      forward RMS 0.87   blame RMS 1.1   (episode: 0.87 and 1.1)


## Section 1: one training step, in numbers
nanoGPT's GPT-2 configuration: 12 windows per micro-batch × 1,024 tokens × 5 accumulation steps × 8 processes. M′ is the number of windows, N′ = M′ · n_max the number of terms. The ` tired` term's blame (−0.90 on its score, Episode 12) enters the average divided by N′.

In [11]:
batch_size, block_size, grad_accum, procs = 12, 1024, 5, 8
tokens_per_iter = batch_size * block_size * grad_accum * procs
M_prime = batch_size * grad_accum * procs
N_prime = M_prime * block_size
print("tokens per iteration:", tokens_per_iter, "| M' =", M_prime, "| N' =", N_prime)
assert tokens_per_iter == 491_520 and M_prime == 480 and N_prime == 491_520 == tokens_per_iter
assert 600_000 * tokens_per_iter == 294_912_000_000                      # by multiplication (the config's comment says 300B)
share = -0.90 / N_prime
print("the ' tired' blame enters the average as", share)
assert round(share * 1e6, 2) == -1.83 and abs(share - (-1.8311e-6)) < 5e-11      # about -1.8e-6

tokens per iteration: 491520 | M' = 480 | N' = 491520
the ' tired' blame enters the average as -1.8310546875000001e-06


## Section 5: warmup then cosine decay
nanoGPT's `get_lr` with learning_rate 6e-4, warmup 2,000 iterations, decay to 6e-5 at iteration 600,000; the cosine part is SGDR's Eq. 5 with T_cur/T_i the fraction of the decay done. Goyal et al.'s warmup (ResNet-50 on images) goes from 0.1 to 3.2: the linear scaling rule with k = 8,192/256 = 32.

In [12]:
learning_rate, min_lr, warmup_iters, lr_decay_iters = 6e-4, 6e-5, 2000, 600_000

def get_lr(it):
    if it < warmup_iters:                                    # linear warmup
        return learning_rate * (it + 1) / (warmup_iters + 1)
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))   # half a cosine wave
    return min_lr + coeff * (learning_rate - min_lr)

print({it: float("%.2g" % get_lr(it)) for it in (0, 1000, 2000, 301_000, 600_000)})
assert abs(get_lr(1000) - 6e-4 * 1001 / 2001) < 1e-15 and float("%.2g" % get_lr(1000)) == 3.0e-4       # the episode's 3.0e-4
assert abs(get_lr(301_000) - (6e-5 + 0.5 * (6e-4 - 6e-5))) < 1e-12 and float("%.2g" % get_lr(301_000)) == 3.3e-4   # and 3.3e-4
assert abs(get_lr(2000) - 6e-4) < 1e-15 and abs(get_lr(600_000) - 6e-5) < 1e-15 and float("%.2g" % get_lr(0)) == 3.0e-7
for it in (2000, 50_000, 301_000, 599_999, 600_000):         # SGDR Eq. 5 gives the same curve
    frac = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    sgdr = min_lr + 0.5 * (learning_rate - min_lr) * (1 + math.cos(math.pi * frac))
    assert abs(sgdr - get_lr(it)) < 1e-15
ups = [get_lr(i) for i in range(0, 2001, 100)]; downs = [get_lr(i) for i in range(2000, 600_001, 20_000)]
assert all(a < b for a, b in zip(ups, ups[1:])) and all(a > b for a, b in zip(downs, downs[1:]))      # up, then down
assert 8192 // 256 == 32 and round(0.1 * 32, 1) == 3.2                                              # Goyal: eta 0.1 -> 3.2

{0: 3e-07, 1000: 0.0003, 2000: 0.0006, 301000: 0.00033, 600000: 6e-05}


## Section 4: weight decay and Adam's memory
For plain SGD, an L2 penalty λ‖θ‖² (Episode 8) changes the update to θ ← (1 − 2ηλ)θ − η · blame, so Loshchilov and Hutter's per-step rate λ equals 2ηλ in Episode 8's notation. Adam keeps two more numbers (m and v) for every parameter; GPT-2 small has 124,439,808 of them.

In [13]:
rng = np.random.default_rng(0)
for _ in range(5):
    th, gr, eta_, lam_ = rng.normal(size=3), rng.normal(size=3), 0.01, 0.05
    assert np.allclose(th - eta_ * (gr + 2 * lam_ * th), (1 - 2 * eta_ * lam_) * th - eta_ * gr)    # L2 in the loss = shrink by 1 - 2*eta*lambda
assert abs(2 * 0.01 * 0.05 - 0.001) < 1e-15

V, d, n_ctx, n_block, d_ff = 50257, 768, 1024, 12, 3072
ln = 2 * d                                                   # one gain and one offset per feature
attn = d * 3 * d + 3 * d + d * d + d
mlp = d * d_ff + d_ff + d_ff * d + d
block = ln + attn + ln + mlp
total = V * d + n_ctx * d + n_block * block + ln
print("GPT-2 small parameters:", total, "| Adam's m and v:", 2 * total)
assert total == 124_439_808 and 2 * total == 248_879_616

GPT-2 small parameters: 124439808 | Adam's m and v: 248879616


## Check question
(a) A parameter's blame is 3 at k = 1 and at k = 2: m̂ and v̂ at k = 2, and Adam's move at η = 0.01. (b) LayerNorm of h = (10, 30, 30, 90) with ω = 0, gain 1 and offset 0.

In [14]:
g = 3.0
m = (1 - RHO1) * g; v = (1 - RHO2) * g * g                   # k = 1
m = RHO1 * m + (1 - RHO1) * g; v = RHO2 * v + (1 - RHO2) * g * g                  # k = 2
mh, vh = m / (1 - RHO1 ** 2), v / (1 - RHO2 ** 2)
step = 0.01 * mh / (math.sqrt(vh) + OMEGA)
print("m", round(m, 6), "m-hat", round(mh, 6), "v", round(v, 6), "v-hat", round(vh, 6), "step", round(step, 6))
assert abs(m - 0.57) < 1e-12 and abs(mh - 3.0) < 1e-12 and abs(v - 0.017991) < 1e-12 and abs(vh - 9.0) < 1e-9
assert abs(step - 0.01) < 1e-9                               # 0.01 * 3/(3 + omega), about 0.01
for g_ in (3.0, -0.5, 8.0):                                  # a constant blame gives m-hat = g and v-hat = g^2 at every step
    m_ = v_ = 0.0
    for k in range(1, 6):
        m_ = RHO1 * m_ + (1 - RHO1) * g_; v_ = RHO2 * v_ + (1 - RHO2) * g_ * g_
        assert abs(m_ / (1 - RHO1 ** k) - g_) < 1e-12 and abs(v_ / (1 - RHO2 ** k) - g_ * g_) < 1e-12
mq, vq, zq = layernorm([10, 30, 30, 90])
print("(b) mean", mq, "var", vq, "output", zq)
assert mq == 40 and vq == 900 and np.allclose(zq, [-1, -1 / 3, -1 / 3, 5 / 3]) and np.allclose(zq, z)     # same as (0, 2, 2, 8)
print("ALL MATH LAB CHECKS PASSED")

m 0.57 m-hat 3.0 v 0.017991 v-hat 9.0 step 0.01
(b) mean 40.0 var 900.0 output [-1.         -0.33333333 -0.33333333  1.66666667]
ALL MATH LAB CHECKS PASSED


## Real scale: GPT-2 small
This cell confirms against the published weights (Hugging Face `openai-community/gpt2`, 548 MB, once): the 11 token IDs of the running sentence (n = 11, N = 10), the parameter count (124,439,808), and the layer-norm numbers (a gain and an offset of 768 each per LayerNorm: 1,536; 12 blocks × 2 + 1 final = 25 norms, 38,400 in all).

In [15]:
import subprocess, sys
for pkg in ["tiktoken", "safetensors", "huggingface_hub"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import tiktoken
from huggingface_hub import hf_hub_download
from safetensors import safe_open

enc = tiktoken.get_encoding("gpt2")
toks = enc.encode("The cat sat on the mat because it was tired.")
print(len(toks), "tokens:", toks)
assert toks == [464, 3797, 3332, 319, 262, 2603, 780, 340, 373, 10032, 13] and enc.decode([toks[9]]) == " tired"
assert len(toks) == 11 and len(toks) - 1 == 10                           # n = 11 tokens, N = 10 terms

path = hf_hub_download("openai-community/gpt2", "model.safetensors", local_dir="gpt2_weights")
fl = safe_open(path, framework="np")
size = lambda name: int(np.prod(tuple(fl.get_slice(name).get_shape())))
keys = list(fl.keys())
n_params = sum(size(k) for k in keys if not k.endswith(".attn.bias"))    # the attention-mask buffers are not parameters
ln_keys = [k for k in keys if ".ln_" in k or k.startswith("ln_f")]
n_ln = sum(size(k) for k in ln_keys)
print("parameters:", n_params, "| layer-norm tensors:", len(ln_keys), "| layer-norm numbers:", n_ln)
assert n_params == total == 124_439_808
assert len(ln_keys) == 50 and n_ln == 38_400 == 25 * 1536               # 25 norms, each a gain and an offset of 768
print("ALL EPISODE 13 CHECKS PASSED")

11 tokens: [464, 3797, 3332, 319, 262, 2603, 780, 340, 373, 10032, 13]


parameters: 124439808 | layer-norm tensors: 50 | layer-norm numbers: 38400
ALL EPISODE 13 CHECKS PASSED
